In [1]:
# Run this once, then comment it out
!pip install faker tqdm --quiet


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [6]:
# =============================================================================
# DIGITAL LENDING: PORTFOLIO OPTIMIZATION
# MODULE 2: DATA ENGINEERING, CLEANING, VALIDATION & FEATURE ENGINEERING
# =============================================================================
# Author   : Senior Fintech ML Engineer & Credit Risk Analytics Architect
# Purpose  : Production-grade data pipeline — cleaning → validation →
#            imputation → outlier handling → feature engineering →
#            aggregation → ML-ready exports
# Tested   : Google Colab & Jupyter Notebook
# Depends  : Output CSVs from Module 1 (lending_data/ folder)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (run once)
# ─────────────────────────────────────────────────────────────────────────────
# Uncomment when running in Google Colab:
# !pip install pandas numpy scipy scikit-learn feature-engine plotly matplotlib \
#              category_encoders tqdm --quiet

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & Global Configuration
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
import logging
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import (LabelEncoder, StandardScaler,
                                   MinMaxScaler, RobustScaler)
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")                  # headless — swap to "inline" in notebooks
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False
    print("⚠  Plotly not found — skipping interactive charts.")

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Config-driven paths ───────────────────────────────────────────────────────
CFG = {
    "input_dir":        "lending_data",          # Module 1 output folder
    "output_dir":       "lending_features",      # Module 2 output folder
    "figures_dir":      "lending_features/figures",
    "log_file":         "lending_features/pipeline.log",
    # IQR multiplier for outlier detection
    "iqr_multiplier":   2.5,
    # KNN imputer neighbours
    "knn_neighbours":   5,
    # Isolation Forest contamination
    "iso_contamination":0.03,
    # Train/test split ratio
    "test_size":        0.20,
    # Stress periods (from Module 1)
    "stress_months": [
        "2022-04", "2022-05", "2022-06", "2022-07",
        "2023-10", "2023-11", "2023-12",
    ],
    # Channel quality scores (higher = better)
    "channel_quality": {
        "Organic Search": 0.85, "App Store": 0.82, "Referral": 0.75,
        "Email Campaign": 0.65, "Social Media": 0.60,
        "DSA Partner": 0.45,   "Bank Branch": 0.50,
    },
}

for d in [CFG["output_dir"], CFG["figures_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Logging setup ─────────────────────────────────────────────────────────────
# Change your log handler block to this:
handlers = [
    logging.FileHandler(CFG["log_file"], mode="w", encoding="utf-8"),
    logging.StreamHandler(sys.stdout)
]

# Ensure the stream handler handles unicode gracefully if the console is stubborn
# Enforce UTF-8 only on the file handler; leave StreamHandler to handle the notebook native outstream
handlers = [
    logging.FileHandler(CFG["log_file"], mode="w", encoding="utf-8"),
    logging.StreamHandler(sys.stdout),
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=handlers,
)
log = logging.getLogger("LendingPipeline")
log.info("Module 2 pipeline started cleanly.")

# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — DATA LOADING LAYER
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

# Expected schemas for validation on load
SCHEMAS = {
    "customers": {
        "customer_id": "object", "age": "int64",
        "monthly_income": "float64", "credit_score": "float64",
        "income_stability_score": "float64",
    },
    "loans": {
        "loan_id": "object", "customer_id": "object",
        "loan_amount": "float64", "interest_rate": "float64",
        "loan_tenure_months": "int64",
    },
    "repayments": {
        "repayment_id": "object", "loan_id": "object",
        "payment_status": "object", "delinquency_stage": "object",
    },
    "behavioral": {
        "signal_id": "object", "customer_id": "object",
        "spending_volatility": "float64",
    },
    "outcomes": {
        "outcome_id": "object", "loan_id": "object",
        "default_flag": "int64", "writeoff_flag": "int64",
    },
}


def load_table(filename: str, date_cols: list = None,
               table_name: str = "") -> pd.DataFrame:
    """
    Load a CSV from the input directory, coerce dtypes, parse dates.
    Raises FileNotFoundError with a helpful message if file is missing.
    """
    path = os.path.join(CFG["input_dir"], filename)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ File not found: {path}\n"
            "   → Run Module 1 first to generate the lending_data/ folder."
        )
    df = pd.read_csv(path, parse_dates=date_cols or [], low_memory=False)
    log.info("Loaded %-35s  rows=%-8d  cols=%d", table_name or filename,
             len(df), df.shape[1])
    return df


def validate_schema(df: pd.DataFrame, schema: dict, table: str) -> list:
    """
    Check that all expected columns exist.
    Returns a list of missing columns.
    """
    missing = [c for c in schema if c not in df.columns]
    if missing:
        log.warning("[%s] Missing columns: %s", table, missing)
    return missing


def check_referential_integrity(child: pd.DataFrame, parent: pd.DataFrame,
                                 fk: str, pk: str,
                                 child_name: str, parent_name: str) -> int:
    """
    Returns the count of FK values in `child` that don't exist in `parent`.
    Logs a warning if any are found.
    """
    orphans = ~child[fk].isin(parent[pk])
    n = orphans.sum()
    if n:
        log.warning("[RI] %d orphan %s values in %s not found in %s.%s",
                    n, fk, child_name, parent_name, pk)
    return n


def load_all_data() -> dict:
    """
    Master loader — loads all 5 tables, validates schemas, checks FK integrity.
    Returns a dict of DataFrames.
    """
    log.info("=" * 60)
    log.info("STEP 1 — DATA LOADING")
    log.info("=" * 60)

    data = {
        "customers":  load_table("01_customer_profile.csv",  table_name="Customer Profile"),
        "loans":      load_table("02_loan_application.csv",
                                 date_cols=["origination_date"],
                                 table_name="Loan Application"),
        "repayments": load_table("03_repayment_behavior.csv",
                                 date_cols=["due_date", "payment_date"],
                                 table_name="Repayment Behavior"),
        "behavioral": load_table("04_behavioral_signals.csv",
                                 date_cols=["month"],
                                 table_name="Behavioral Signals"),
        "outcomes":   load_table("05_outcome.csv",            table_name="Outcomes"),
    }

    # Schema checks
    for tname, schema in SCHEMAS.items():
        validate_schema(data[tname], schema, tname)

    # Referential integrity
    check_referential_integrity(
        data["loans"], data["customers"], "customer_id", "customer_id",
        "loans", "customers"
    )
    check_referential_integrity(
        data["repayments"], data["loans"], "loan_id", "loan_id",
        "repayments", "loans"
    )
    check_referential_integrity(
        data["outcomes"], data["loans"], "loan_id", "loan_id",
        "outcomes", "loans"
    )

    log.info("Data loading complete.")
    return data


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — DATA CLEANING LAYER
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def remove_duplicates(df: pd.DataFrame, pk: str, table: str) -> pd.DataFrame:
    """Drop duplicate rows on primary key; log count removed."""
    n_before = len(df)
    df = df.drop_duplicates(subset=[pk], keep="first")
    dropped = n_before - len(df)
    if dropped:
        log.warning("[%s] Removed %d duplicate %s rows.", table, dropped, pk)
    return df


def fix_customer_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Customer-specific cleaning:
    • Age must be 18-80
    • Income must be positive
    • Credit score clamped to 300-900
    • Standardise categorical strings
    """
    n = len(df)
    # Age filter
    df = df[df["age"].between(18, 80)].copy()
    log.info("[Customers] Removed %d rows with invalid age.", n - len(df))

    # Income — replace negatives/zero with NaN (will be imputed later)
    neg_inc = (df["monthly_income"] <= 0).sum()
    df.loc[df["monthly_income"] <= 0, "monthly_income"] = np.nan
    if neg_inc:
        log.warning("[Customers] %d non-positive income values → NaN.", neg_inc)

    # Credit score clamp
    df["credit_score"] = df["credit_score"].clip(300, 900)

    # Standardise string columns
    str_cols = ["gender", "employment_type", "education_level",
                "marital_status", "urban_semiurban_flag",
                "device_type", "acquisition_channel"]
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()

    return df


def fix_loan_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Loan-specific cleaning:
    • Loan amount must be > 0
    • Interest rate clamped 5–50%
    • Tenure > 0
    • EMI >= 0
    """
    # Negative/zero loan amounts → NaN
    bad_amt = (df["loan_amount"] <= 0).sum()
    df.loc[df["loan_amount"] <= 0, "loan_amount"] = np.nan
    if bad_amt:
        log.warning("[Loans] %d non-positive loan_amount → NaN.", bad_amt)

    df["interest_rate"]      = df["interest_rate"].clip(5, 50)
    df["loan_tenure_months"] = df["loan_tenure_months"].clip(1, 120)
    df["emi_amount"]         = df["emi_amount"].clip(lower=0)

    # Origination date sanity
    if "origination_date" in df.columns:
        df["origination_date"] = pd.to_datetime(df["origination_date"], errors="coerce")
        bad_dates = df["origination_date"].isna().sum()
        if bad_dates:
            log.warning("[Loans] %d unparseable origination_date.", bad_dates)

    return df


def fix_repayment_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Repayment cleaning:
    • delay_days cannot be negative
    • bounce_count cannot be negative
    • Validate delinquency stage values
    """
    valid_stages = {"Current", "DPD_30", "DPD_60", "DPD_90", "Write-Off"}
    invalid_stage = ~df["delinquency_stage"].isin(valid_stages)
    df.loc[invalid_stage, "delinquency_stage"] = "Current"
    if invalid_stage.sum():
        log.warning("[Repayments] %d invalid delinquency_stage → 'Current'.",
                    invalid_stage.sum())

    df["delay_days"]   = df["delay_days"].clip(lower=0)
    df["bounce_count"] = df["bounce_count"].clip(lower=0)

    return df


def clean_all(data: dict) -> dict:
    """Apply all cleaning functions to all tables."""
    log.info("=" * 60)
    log.info("STEP 2 — DATA CLEANING")
    log.info("=" * 60)

    data["customers"]  = remove_duplicates(data["customers"],  "customer_id", "customers")
    data["loans"]      = remove_duplicates(data["loans"],      "loan_id",     "loans")
    data["repayments"] = remove_duplicates(data["repayments"], "repayment_id","repayments")
    data["behavioral"] = remove_duplicates(data["behavioral"], "signal_id",   "behavioral")
    data["outcomes"]   = remove_duplicates(data["outcomes"],   "outcome_id",  "outcomes")

    data["customers"]  = fix_customer_data(data["customers"])
    data["loans"]      = fix_loan_data(data["loans"])
    data["repayments"] = fix_repayment_data(data["repayments"])

    log.info("Data cleaning complete.")
    return data


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — DATA VALIDATION LAYER
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def run_validation(data: dict) -> pd.DataFrame:
    """
    Run a battery of validation checks.
    Returns a DataFrame summarising check name, result, count, severity.
    """
    log.info("=" * 60)
    log.info("STEP 3 -- DATA VALIDATION")
    log.info("=" * 60)

    results = []

    # Clean, explicitly 4-space indented inner function
    def record(check, passed, count=0, severity="INFO"):
        status = "PASS" if passed else "FAIL"
        results.append({
            "check":    check,
            "status":   status,
            "count":    count,
            "severity": severity,
        })
        symbol = "[OK]  " if passed else ("[FAIL]" if severity == "CRITICAL" else "[WARN]")
        log.info("  %s %-55s count=%d", symbol, check, count)

    c = data["customers"]
    l = data["loans"]
    r = data["repayments"]
    o = data["outcomes"]

    # 1. Age range
    bad_age = (~c["age"].between(18, 80)).sum()
    record("Customer age in [18, 80]", bad_age == 0, bad_age, "CRITICAL")

    # 2. Credit score range
    bad_cs = (~c["credit_score"].between(300, 900)).sum()
    record("Credit score in [300, 900]", bad_cs == 0, bad_cs, "CRITICAL")

    # 3. Loan amount > 0
    bad_la = (l["loan_amount"] <= 0).sum()
    record("Loan amount > 0", bad_la == 0, bad_la, "CRITICAL")

    # 4. Interest rate sanity
    bad_ir = (~l["interest_rate"].between(5, 50)).sum()
    record("Interest rate in [5%, 50%]", bad_ir == 0, bad_ir, "HIGH")

    # 5. EMI > 0 for approved loans
    approved = l[l["approval_status"] == "Approved"]
    bad_emi  = (approved["emi_amount"] <= 0).sum()
    record("EMI > 0 for approved loans", bad_emi == 0, bad_emi, "HIGH")

    # 6. EMI consistency (EMI <= loan_amount check)
    emi_exceed = (approved["emi_amount"] > approved["loan_amount"]).sum()
    record("EMI <= loan_amount", emi_exceed == 0, emi_exceed, "MEDIUM")

    # 7. Repayments only for existing loans
    orphan_rep = (~r["loan_id"].isin(l["loan_id"])).sum()
    record("No orphan repayment records", orphan_rep == 0, orphan_rep, "CRITICAL")

    # 8. Delinquency stage values
    valid_stages = {"Current", "DPD_30", "DPD_60", "DPD_90", "Write-Off"}
    bad_stage = (~r["delinquency_stage"].isin(valid_stages)).sum()
    record("Valid delinquency_stage values", bad_stage == 0, bad_stage, "HIGH")

    # 9. Default flag binary
    bad_def = (~o["default_flag"].isin([0, 1])).sum()
    record("default_flag is binary (0/1)", bad_def == 0, bad_def, "CRITICAL")

    # 10. Default rate reasonable
    dr = o["default_flag"].mean() * 100
    record(f"Default rate in [5%, 25%] (actual={dr:.1f}%)",
           5 <= dr <= 25, 0 if 5 <= dr <= 25 else 1, "HIGH")

    # 11. Duplicate loan IDs
    dup_loans = l["loan_id"].duplicated().sum()
    record("No duplicate loan_id", dup_loans == 0, dup_loans, "CRITICAL")

    # 12. Missing critical customer fields
    for col in ["customer_id", "employment_type", "loan_type"]:
        tbl, series = ("customers", c) if col in c.columns else ("loans", l)
        miss = series[col].isna().sum() if col in series.columns else 0
        record(f"No missing values in {tbl}.{col}", miss == 0, miss, "HIGH")

    val_df = pd.DataFrame(results)
    fails  = val_df[val_df["status"] == "FAIL"]
    log.info("Validation complete: %d checks | %d FAILED", len(val_df), len(fails))
    return val_df


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — MISSING VALUE STRATEGY
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def missing_value_report(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Return a DataFrame showing missing counts and percentages per column."""
    miss  = df.isna().sum()
    pct   = miss / len(df) * 100
    report = pd.DataFrame({"missing_count": miss, "missing_pct": pct})
    report = report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)
    if not report.empty:
        log.info("[%s] Missing value summary:\n%s", table,
                 report.to_string())
    return report


def impute_customers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Intelligent imputation for customer profiles.

    Strategy reasoning:
    • monthly_income    → median by employment_type (MAR: missingness depends
                          on employment; median is robust to skew)
    • bank_balance_avg  → median by employment_type (correlated with income)
    • credit_history_length → median by age_bucket (MAR: younger = less history)
    • income_stability_score → KNN on [credit_score, monthly_income, age]
    • credit_score      → KNN on [monthly_income, credit_history_length, age]

    Missing indicator columns are added so models can learn from
    the missingness pattern itself.
    """
    log.info("[Imputation] Customer table …")

    # ── Add missing indicator flags first ────────────────────────────────────
    for col in ["monthly_income", "credit_score",
                "income_stability_score", "bank_balance_avg"]:
        if col in df.columns:
            df[f"{col}_was_missing"] = df[col].isna().astype(int)

    # ── Age bucket for conditional imputation ─────────────────────────────────
    df["age_bucket"] = pd.cut(df["age"].fillna(35),
                               bins=[17, 25, 35, 45, 55, 80],
                               labels=["18-25", "26-35", "36-45", "46-55", "56+"])

    # ── Median by employment type ─────────────────────────────────────────────
    for col in ["monthly_income", "bank_balance_avg"]:
        if col in df.columns:
            group_medians = df.groupby("employment_type")[col].transform("median")
            df[col] = df[col].fillna(group_medians)
            # Fallback to global median if group median also NaN
            df[col] = df[col].fillna(df[col].median())

    # ── Conditional imputation: credit_history by age bucket ─────────────────
    if "credit_history_length" in df.columns:
        df["credit_history_length"] = df.groupby("age_bucket")[
            "credit_history_length"
        ].transform(lambda x: x.fillna(x.median()))
        df["credit_history_length"] = df["credit_history_length"].fillna(0)

    # ── KNN imputation for score columns ─────────────────────────────────────
    knn_cols = ["credit_score", "income_stability_score"]
    knn_feat = ["monthly_income", "credit_history_length", "age",
                "digital_engagement_score"]
    knn_feat = [c for c in knn_feat if c in df.columns]
    knn_df   = df[knn_cols + knn_feat].copy()

    # Scale before KNN
    scaler   = RobustScaler()
    knn_arr  = scaler.fit_transform(knn_df)
    imputer  = KNNImputer(n_neighbors=CFG["knn_neighbours"])
    knn_imp  = imputer.fit_transform(knn_arr)
    knn_imp_df = pd.DataFrame(knn_imp, columns=knn_cols + knn_feat,
                               index=df.index)
    for col in knn_cols:
        df[col] = df[col].fillna(knn_imp_df[col])

    df.drop(columns=["age_bucket"], inplace=True, errors="ignore")
    log.info("[Imputation] Customer imputation complete.")
    return df


def impute_loans(df: pd.DataFrame) -> pd.DataFrame:
    """
    Loan-level imputation.
    • loan_amount  → median by loan_type
    • processing_time → median by risk grade
    • acquisition_cost → median by acquisition_channel
    """
    log.info("[Imputation] Loan table …")

    for col in ["loan_amount", "emi_amount"]:
        df[f"{col}_was_missing"] = df[col].isna().astype(int)

    for col, groupby in [
        ("loan_amount",        "loan_type"),
        ("processing_time_hours", "origination_risk_grade"),
        ("acquisition_cost",   "acquisition_channel" if "acquisition_channel" in df.columns else "loan_type"),
    ]:
        if col in df.columns and groupby in df.columns:
            df[col] = df[col].fillna(
                df.groupby(groupby)[col].transform("median")
            ).fillna(df[col].median())

    log.info("[Imputation] Loan imputation complete.")
    return df


def impute_behavioral(df: pd.DataFrame) -> pd.DataFrame:
    """
    Behavioral signals imputation.
    app_usage_frequency → missing often means no app usage → fill with 0
    (business-driven MCAR: offline customers just don't use the app)
    """
    if "app_usage_frequency" in df.columns:
        df["app_usage_was_missing"] = df["app_usage_frequency"].isna().astype(int)
        df["app_usage_frequency"]   = df["app_usage_frequency"].fillna(0)
    return df


def run_imputation(data: dict) -> dict:
    """Run all imputation strategies."""
    log.info("=" * 60)
    log.info("STEP 4 — MISSING VALUE IMPUTATION")
    log.info("=" * 60)

    for tbl in ["customers", "loans", "repayments", "behavioral", "outcomes"]:
        missing_value_report(data[tbl], tbl)

    data["customers"]  = impute_customers(data["customers"])
    data["loans"]      = impute_loans(data["loans"])
    data["behavioral"] = impute_behavioral(data["behavioral"])

    log.info("Imputation complete.")
    return data


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — OUTLIER DETECTION
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def iqr_bounds(series: pd.Series, k: float = None):
    """Return (lower, upper) IQR-based bounds."""
    k = k or CFG["iqr_multiplier"]
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - k * IQR, Q3 + k * IQR


def flag_outliers_iqr(df: pd.DataFrame, col: str, flag_col: str = None) -> pd.DataFrame:
    """Add a binary flag column for IQR outliers. Does NOT remove them."""
    flag = flag_col or f"{col}_outlier_flag"
    lo, hi = iqr_bounds(df[col].dropna())
    df[flag] = ((df[col] < lo) | (df[col] > hi)).astype(int)
    n = df[flag].sum()
    if n:
        log.info("[Outliers IQR] %-35s  flagged=%d", col, n)
    return df


def flag_outliers_zscore(df: pd.DataFrame, col: str, threshold: float = 3.5) -> pd.DataFrame:
    """Add a z-score outlier flag. Robust to skewed distributions."""
    flag = f"{col}_zscore_flag"
    z = np.abs(stats.zscore(df[col].fillna(df[col].median())))
    df[flag] = (z > threshold).astype(int)
    return df


def isolation_forest_anomaly(df: pd.DataFrame, cols: list,
                               flag_col: str = "iso_anomaly_flag") -> pd.DataFrame:
    """
    Fit an Isolation Forest on the given columns.
    Marks -1 samples (anomalies) as 1 in the flag column.
    Business interpretation: these may be fraud, data entry errors,
    or genuinely unusual (investigate before removing).
    """
    log.info("[Outliers ISO] Running Isolation Forest on %s …", cols)
    sub = df[cols].fillna(df[cols].median())
    iso = IsolationForest(
        contamination=CFG["iso_contamination"],
        random_state=SEED, n_jobs=-1
    )
    df[flag_col] = (iso.fit_predict(sub) == -1).astype(int)
    log.info("[Outliers ISO] Flagged %d anomalies.", df[flag_col].sum())
    return df


def winsorise(series: pd.Series, lo_pct: float = 0.01,
              hi_pct: float = 0.99) -> pd.Series:
    """
    Percentile clipping (Winsorisation).
    Used for ML feature preparation — replaces extremes with boundary values
    rather than removing them (preserves row count).
    """
    return series.clip(series.quantile(lo_pct), series.quantile(hi_pct))


def run_outlier_detection(data: dict) -> dict:
    """Apply outlier detection across key columns."""
    log.info("=" * 60)
    log.info("STEP 5 — OUTLIER DETECTION")
    log.info("=" * 60)

    c = data["customers"]
    l = data["loans"]

    # Customer-level
    for col in ["monthly_income", "credit_score", "bank_balance_avg"]:
        if col in c.columns:
            c = flag_outliers_iqr(c, col)

    # Isolation Forest on customer financials
    iso_cols = [col for col in
                ["monthly_income", "credit_score", "bank_balance_avg",
                 "credit_history_length", "digital_engagement_score"]
                if col in c.columns]
    c = isolation_forest_anomaly(c, iso_cols, "customer_iso_anomaly")

    # Loan-level
    for col in ["loan_amount", "interest_rate", "emi_amount"]:
        if col in l.columns:
            l = flag_outliers_iqr(l, col)

    data["customers"] = c
    data["loans"]     = l
    log.info("Outlier detection complete.")
    return data


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — FEATURE ENGINEERING LAYER
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def engineer_customer_risk_features(customers: pd.DataFrame,
                                     loans: pd.DataFrame) -> pd.DataFrame:
    """
    Build customer-level risk features from the customer and loan tables.

    Feature dictionary:
    ─────────────────────────────────────────────────────────────────
    debt_to_income_ratio     : Total outstanding loan balance / monthly income.
                               High DTI → financial overextension → default risk.
    emi_to_income_ratio      : Average EMI / monthly income.
                               > 0.4 is generally considered high-stress.
    utilization_ratio        : Existing loans / (credit history + 1).
                               Proxy for credit utilisation.
    leverage_score           : Normalised measure of total debt burden.
    financial_stress_index   : Composite of DTI, EMI ratio, existing_loans, volatility.
    credit_stability_index   : Inverse of volatility indicators; high = stable.
    income_volatility_proxy  : Estimated from employment type stability.
    digital_trust_score      : Composite of engagement, device quality, channel.
    ─────────────────────────────────────────────────────────────────
    """
    log.info("[Features] Engineering customer risk features …")

    # ── Loan-level aggregations per customer ──────────────────────────────────
    approved = loans[loans["approval_status"] == "Approved"].copy()
    cust_loan = approved.groupby("customer_id").agg(
        total_loan_balance  = ("loan_amount",  "sum"),
        avg_emi             = ("emi_amount",   "mean"),
        loan_count          = ("loan_id",      "count"),
        avg_interest_rate   = ("interest_rate","mean"),
    ).reset_index()

    c = customers.merge(cust_loan, on="customer_id", how="left")
    c["total_loan_balance"] = c["total_loan_balance"].fillna(0)
    c["avg_emi"]            = c["avg_emi"].fillna(0)
    c["loan_count"]         = c["loan_count"].fillna(0)
    c["avg_interest_rate"]  = c["avg_interest_rate"].fillna(0)

    # ── Core risk ratios ──────────────────────────────────────────────────────
    inc = c["monthly_income"].replace(0, np.nan)
    c["debt_to_income_ratio"]  = (c["total_loan_balance"] / (inc * 12)).round(4)
    c["emi_to_income_ratio"]   = (c["avg_emi"] / inc).round(4)
    c["utilization_ratio"]     = (
        c["existing_loans"] / (c["credit_history_length"] + 1)
    ).round(4)

    # Leverage score: normalised 0–1
    c["leverage_score"] = (
        0.4 * c["debt_to_income_ratio"].clip(0, 5) / 5
        + 0.4 * c["emi_to_income_ratio"].clip(0, 1)
        + 0.2 * c["utilization_ratio"].clip(0, 1)
    ).round(4)

    # ── Financial stress index ────────────────────────────────────────────────
    c["financial_stress_index"] = (
        0.30 * c["emi_to_income_ratio"].clip(0, 1)
        + 0.25 * (1 - c["income_stability_score"].fillna(0.5))
        + 0.25 * c["leverage_score"]
        + 0.20 * (c["existing_loans"].clip(0, 5) / 5)
    ).round(4)

    # ── Credit stability index (higher = more stable) ─────────────────────────
    norm_cs = (c["credit_score"].fillna(600) - 300) / 600
    c["credit_stability_index"] = (
        0.40 * norm_cs
        + 0.30 * c["income_stability_score"].fillna(0.5)
        + 0.30 * (c["credit_history_length"].fillna(0).clip(0, 120) / 120)
    ).round(4)

    # ── Income volatility proxy ───────────────────────────────────────────────
    vol_map = {
        "Salaried": 0.10, "Self-Employed": 0.40,
        "Gig Worker": 0.65, "First-Time Borrower": 0.35,
        "Sme Owner": 0.45, "Sme owner": 0.45,
    }
    c["income_volatility_proxy"] = (
        c["employment_type"].map(vol_map).fillna(0.30)
        + np.random.normal(0, 0.05, len(c))
    ).clip(0, 1).round(4)

    # ── Digital trust score ───────────────────────────────────────────────────
    channel_quality_map = CFG["channel_quality"]
    c["channel_quality_score"] = c["acquisition_channel"].map(
        channel_quality_map
    ).fillna(0.55)
    device_score = c["device_type"].map(
        {"Ios": 0.9, "Android": 0.85, "Desktop": 0.70, "Feature Phone": 0.30}
    ).fillna(0.60)
    c["digital_trust_score"] = (
        0.40 * (c["digital_engagement_score"].fillna(50) / 100)
        + 0.35 * device_score
        + 0.25 * c["channel_quality_score"]
    ).round(4)

    log.info("[Features] Customer risk features done.")
    return c


def engineer_repayment_features(repayments: pd.DataFrame) -> pd.DataFrame:
    """
    Loan-level repayment behaviour features.

    Feature dictionary:
    ─────────────────────────────────────────────────────────────────
    avg_delay_days             : Average days late per instalment.
    max_delay_days             : Worst single delayed payment.
    missed_payment_ratio       : Fraction of instalments missed.
    delinquency_frequency      : Fraction of instalments with DPD > 0.
    bounce_frequency           : Average bounce count per instalment.
    payment_regularization_score: 1 minus missed_payment_ratio (higher=better).
    recovery_ratio             : On-time payments / total payments.
    worst_delinquency_stage    : Ordinal encoding of worst-ever stage.
    rolling_dpd_trend          : Average delay in last 3 instalments (recency).
    ─────────────────────────────────────────────────────────────────
    """
    log.info("[Features] Engineering repayment features …")

    r = repayments.copy()
    r["delay_days"]  = r["delay_days"].fillna(0)
    r["is_late"]     = (r["payment_status"].isin(
        ["Late", "Partial", "Missed"])).astype(int)
    r["is_missed"]   = (r["payment_status"] == "Missed").astype(int)
    r["is_on_time"]  = (r["payment_status"] == "On-Time").astype(int)

    # Sort by loan and due date for rolling features
    r = r.sort_values(["loan_id", "due_date"])

    # ── Rolling last-3 delay trend ────────────────────────────────────────────
    r["rolling_3_delay"] = (
        r.groupby("loan_id")["delay_days"]
         .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

    # ── Delinquency stage ordinal ─────────────────────────────────────────────
    stage_ord = {"Current": 0, "DPD_30": 1, "DPD_60": 2,
                 "DPD_90": 3, "Write-Off": 4}
    r["stage_ordinal"] = r["delinquency_stage"].map(stage_ord).fillna(0)

    # ── Aggregate per loan ────────────────────────────────────────────────────
    feat = r.groupby("loan_id").agg(
        avg_delay_days         = ("delay_days",    "mean"),
        max_delay_days         = ("delay_days",    "max"),
        missed_payment_ratio   = ("is_missed",     "mean"),
        delinquency_frequency  = ("is_late",       "mean"),
        bounce_frequency       = ("bounce_count",  "mean"),
        total_payments         = ("repayment_id",  "count"),
        on_time_count          = ("is_on_time",    "sum"),
        worst_delinquency_stage= ("stage_ordinal", "max"),
        rolling_dpd_trend      = ("rolling_3_delay","last"),  # most recent rolling avg
    ).reset_index()

    feat["payment_regularization_score"] = (
        1 - feat["missed_payment_ratio"]
    ).clip(0, 1).round(4)
    feat["recovery_ratio"] = (
        feat["on_time_count"] / feat["total_payments"].replace(0, 1)
    ).round(4)

    for col in ["avg_delay_days", "max_delay_days", "rolling_dpd_trend",
                "bounce_frequency"]:
        feat[col] = feat[col].round(2)

    log.info("[Features] Repayment features done.  Shape: %s", feat.shape)
    return feat


def engineer_behavioral_features(behavioral: pd.DataFrame) -> pd.DataFrame:
    """
    Customer-level behavioral intelligence features aggregated across months.

    Feature dictionary:
    ─────────────────────────────────────────────────────────────────
    spending_volatility_index    : Mean spending volatility over observation window.
    balance_instability_score    : Mean balance volatility.
    transaction_irregularity_score: Coefficient of variation of transaction frequency.
    spending_shock_frequency     : Fraction of months with spending shock.
    nighttime_risk_score         : Mean late-night transaction ratio.
    mobility_instability_score   : Mean location change frequency (normalised).
    behavioral_deterioration_score: Trend slope of spending_volatility over time
                                   (positive = worsening).
    ─────────────────────────────────────────────────────────────────
    """
    log.info("[Features] Engineering behavioral features …")

    b = behavioral.sort_values(["customer_id", "month"])

    # ── Trend: linear slope of spending_volatility over time ─────────────────
    def trend_slope(series):
        if len(series) < 3:
            return 0.0
        x = np.arange(len(series))
        slope, *_ = np.polyfit(x, series.values, 1)
        return round(float(slope), 6)

    deterioration = (
        b.groupby("customer_id")["spending_volatility"]
         .apply(trend_slope)
         .reset_index()
         .rename(columns={"spending_volatility": "behavioral_deterioration_score"})
    )

    # ── Aggregate per customer ────────────────────────────────────────────────
    feat = b.groupby("customer_id").agg(
        spending_volatility_index       = ("spending_volatility",      "mean"),
        balance_instability_score       = ("balance_volatility",       "mean"),
        txn_freq_mean                   = ("transaction_frequency",    "mean"),
        txn_freq_std                    = ("transaction_frequency",    "std"),
        spending_shock_frequency        = ("spending_shock_flag",      "mean"),
        nighttime_risk_score            = ("late_night_transaction_ratio","mean"),
        cashflow_consistency_score_mean = ("cashflow_consistency_score","mean"),
        app_usage_mean                  = ("app_usage_frequency",      "mean"),
        location_change_mean            = ("location_change_frequency","mean"),
        obs_months                      = ("month",                    "count"),
    ).reset_index()

    feat["transaction_irregularity_score"] = (
        feat["txn_freq_std"] / feat["txn_freq_mean"].replace(0, 1)
    ).fillna(0).round(4)
    feat["mobility_instability_score"] = (
        feat["location_change_mean"].clip(0, 10) / 10
    ).round(4)

    feat = feat.merge(deterioration, on="customer_id", how="left")
    feat["behavioral_deterioration_score"] = feat[
        "behavioral_deterioration_score"
    ].fillna(0)

    feat.drop(columns=["txn_freq_mean", "txn_freq_std"], inplace=True)
    log.info("[Features] Behavioral features done.  Shape: %s", feat.shape)
    return feat


def engineer_temporal_features(loans: pd.DataFrame,
                                repayments: pd.DataFrame) -> pd.DataFrame:
    """
    Temporal and cohort-level features for the loan table.

    Feature dictionary:
    ─────────────────────────────────────────────────────────────────
    loan_age_days              : Days since origination (snapshot = today).
    cohort_month               : Year-Month of origination (for cohort analysis).
    origination_quarter        : Quarter of origination.
    is_stress_period           : 1 if originated during macro-stress months.
    repayment_age_days         : Days from first to last repayment record.
    ─────────────────────────────────────────────────────────────────
    """
    log.info("[Features] Engineering temporal features …")

    l = loans.copy()
    snapshot = pd.Timestamp("2025-01-01")
    l["loan_age_days"] = (snapshot - l["origination_date"]).dt.days
    l["cohort_month"]  = l["origination_date"].dt.to_period("M").astype(str)
    l["origination_quarter"] = (
        l["origination_date"].dt.year.astype(str)
        + "Q"
        + l["origination_date"].dt.quarter.astype(str)
    )

    stress_set = set(CFG["stress_months"])
    l["is_stress_period"] = l["cohort_month"].isin(stress_set).astype(int)

    # ── Repayment age: first to last payment date per loan ────────────────────
    r_dates = repayments.dropna(subset=["payment_date"]).groupby("loan_id").agg(
        first_payment = ("payment_date", "min"),
        last_payment  = ("payment_date", "max"),
    ).reset_index()
    r_dates["repayment_age_days"] = (
        r_dates["last_payment"] - r_dates["first_payment"]
    ).dt.days
    l = l.merge(r_dates[["loan_id", "repayment_age_days"]], on="loan_id", how="left")

    log.info("[Features] Temporal features done.")
    return l


def engineer_portfolio_features(loans: pd.DataFrame,
                                  outcomes: pd.DataFrame) -> pd.DataFrame:
    """
    Loan-level portfolio and profitability features.

    Feature dictionary:
    ─────────────────────────────────────────────────────────────────
    expected_loss              : loan_amount × PD_proxy × LGD_proxy
    probability_of_default_proxy: Risk grade → empirical PD
    exposure_at_default        : Simplified: outstanding balance at risk
    loss_given_default_proxy   : 1 - recovery_amount / loan_amount
    acquisition_efficiency_score: CLV / acquisition_cost
    ─────────────────────────────────────────────────────────────────
    """
    log.info("[Features] Engineering portfolio features …")

    l = loans.merge(
        outcomes[["loan_id", "default_flag", "writeoff_flag",
                  "recovery_amount", "customer_lifetime_value",
                  "profitability_score", "risk_adjusted_return"]],
        on="loan_id", how="left"
    )

    grade_pd = {"A": 0.03, "B": 0.07, "C": 0.13, "D": 0.22, "E": 0.32}
    l["probability_of_default_proxy"] = l["origination_risk_grade"].map(
        grade_pd
    ).fillna(0.15)

    l["exposure_at_default"] = l["loan_amount"]   # simplified; full balance
    l["loss_given_default_proxy"] = (
        1 - l["recovery_amount"].fillna(0) / l["loan_amount"].replace(0, np.nan)
    ).clip(0, 1).round(4)

    l["expected_loss"] = (
        l["probability_of_default_proxy"]
        * l["exposure_at_default"]
        * l["loss_given_default_proxy"]
    ).round(2)

    acq_cost = l["acquisition_cost"].fillna(500).replace(0, 1)
    l["acquisition_efficiency_score"] = (
        l["customer_lifetime_value"].fillna(0) / acq_cost
    ).round(4)

    log.info("[Features] Portfolio features done.")
    return l


def engineer_early_warning_features(customers: pd.DataFrame,
                                      rep_feats: pd.DataFrame,
                                      beh_feats: pd.DataFrame) -> pd.DataFrame:
    """
    Combine repayment and behavioural signals into an
    early_warning_risk_score per customer.

    Signals used:
    • rolling_dpd_trend           → rising DPD is a primary EWS signal
    • behavioral_deterioration_score → worsening spending patterns
    • spending_shock_frequency    → sudden cashflow disruptions
    • missed_payment_ratio        → history of non-payment
    • cashflow_consistency_score_mean → low consistency = risk
    • app_usage_mean              → reduced engagement predicts default
    """
    log.info("[Features] Engineering early warning features …")

    # Join repayment features back to loans, then to customers
    # rep_feats is loan-level — need customer-level max/mean
    # First, get loan→customer mapping
    pass  # handled in assemble_master_table() below

    ew = customers.copy()

    # Merge behavioural
    ew = ew.merge(beh_feats[[
        "customer_id", "behavioral_deterioration_score",
        "spending_shock_frequency", "cashflow_consistency_score_mean",
        "app_usage_mean", "spending_volatility_index",
        "balance_instability_score",
    ]], on="customer_id", how="left")

    # Early warning composite (all components 0–1; higher = riskier)
    ew["early_warning_risk_score"] = (
        0.25 * ew["spending_volatility_index"].fillna(0.3)
        + 0.20 * ew["behavioral_deterioration_score"].clip(0, 0.1) / 0.1
        + 0.20 * ew["spending_shock_frequency"].fillna(0)
        + 0.20 * (1 - ew["cashflow_consistency_score_mean"].fillna(0.5))
        + 0.15 * (1 - (ew["app_usage_mean"].fillna(10).clip(0, 50) / 50))
    ).clip(0, 1).round(4)

    log.info("[Features] Early warning features done.")
    return ew


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — AGGREGATION & MASTER TABLE ASSEMBLY
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def assemble_master_table(data: dict,
                           rep_feats: pd.DataFrame,
                           beh_feats: pd.DataFrame,
                           loan_feats: pd.DataFrame) -> pd.DataFrame:
    """
    Build the master loan-level feature table by joining:
    Customer features → Loan features → Repayment features → Behavioral features

    This is the primary input for ML models.
    """
    log.info("[Assembly] Building master loan-level feature table …")

    # Start with loan + portfolio features
    master = loan_feats.copy()

    # Join customer-level features
    cust_cols = [
        "customer_id", "age", "gender", "employment_type",
        "monthly_income", "income_stability_score",
        "credit_score", "credit_history_length",
        "bank_balance_avg", "existing_loans",
        "urban_semiurban_flag", "digital_engagement_score",
        "debt_to_income_ratio", "emi_to_income_ratio",
        "leverage_score", "financial_stress_index",
        "credit_stability_index", "income_volatility_proxy",
        "digital_trust_score", "channel_quality_score",
        "customer_iso_anomaly",
        "early_warning_risk_score",
    ]
    cust_cols = [c for c in cust_cols if c in data["customers"].columns]
    master = master.merge(data["customers"][cust_cols],
                          on="customer_id", how="left")

    # Join repayment features (loan-level)
    master = master.merge(rep_feats, on="loan_id", how="left")

    # Join behavioral features (customer-level) — avg signals for this customer
    beh_cols = [
        "customer_id", "spending_volatility_index",
        "balance_instability_score", "transaction_irregularity_score",
        "spending_shock_frequency", "nighttime_risk_score",
        "mobility_instability_score", "behavioral_deterioration_score",
        "cashflow_consistency_score_mean", "app_usage_mean",
    ]
    beh_cols = [c for c in beh_cols if c in beh_feats.columns]
    master = master.merge(beh_feats[beh_cols], on="customer_id", how="left")

    log.info("[Assembly] Master table shape: %s", master.shape)
    return master


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — PORTFOLIO KPI ENGINE
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def compute_portfolio_kpis(master: pd.DataFrame) -> dict:
    """
    Compute executive-level portfolio KPIs from the master table.

    Returns a dict of KPI DataFrames:
    • overall_kpis       : Single-row summary metrics
    • by_loan_type       : KPIs broken down by loan product
    • by_risk_grade      : KPIs by origination risk grade
    • by_channel         : Performance by acquisition channel
    • by_cohort_quarter  : Quarterly cohort performance
    """
    log.info("[KPIs] Computing portfolio KPIs …")

    approved = master[master["approval_status"] == "Approved"].copy()

    def kpi_group(df):
        total_loans     = len(df)
        total_disbursed = df["loan_amount"].sum()
        defaults        = df["default_flag"].sum() if "default_flag" in df.columns else np.nan
        writeoffs       = df["writeoff_flag"].sum() if "writeoff_flag" in df.columns else np.nan
        recovery        = df["recovery_amount"].sum() if "recovery_amount" in df.columns else np.nan
        total_el        = df["expected_loss"].sum() if "expected_loss" in df.columns else np.nan
        avg_int_rate    = df["interest_rate"].mean()
        avg_rar         = df["risk_adjusted_return"].mean() if "risk_adjusted_return" in df.columns else np.nan
        avg_clv         = df["customer_lifetime_value"].mean() if "customer_lifetime_value" in df.columns else np.nan
        avg_acq_cost    = df["acquisition_cost"].mean() if "acquisition_cost" in df.columns else np.nan

        dr  = defaults / total_loans if total_loans else 0
        wor = writeoffs / total_loans if total_loans else 0
        npa = (defaults / total_disbursed * 100) if total_disbursed else 0
        col_eff = 1 - wor

        return {
            "total_loans":           total_loans,
            "total_disbursed_INR":   round(total_disbursed, 0),
            "default_rate_pct":      round(dr * 100, 2),
            "writeoff_rate_pct":     round(wor * 100, 2),
            "npa_ratio_pct":         round(npa, 2),
            "collection_efficiency": round(col_eff * 100, 2),
            "total_expected_loss":   round(total_el, 0),
            "recovery_amount":       round(recovery, 0),
            "avg_interest_rate":     round(avg_int_rate, 2),
            "avg_risk_adj_return":   round(avg_rar, 4),
            "avg_clv":               round(avg_clv, 2),
            "avg_acquisition_cost":  round(avg_acq_cost, 2),
        }

    overall = pd.DataFrame([kpi_group(approved)])
    overall.insert(0, "segment", "Overall Portfolio")

    by_loan_type = approved.groupby("loan_type").apply(kpi_group).apply(
        pd.Series
    ).reset_index()
    by_loan_type.rename(columns={"loan_type": "segment"}, inplace=True)

    by_risk = approved.groupby("origination_risk_grade").apply(kpi_group).apply(
        pd.Series
    ).reset_index()
    by_risk.rename(columns={"origination_risk_grade": "segment"}, inplace=True)

    by_channel_col = "acquisition_channel" if "acquisition_channel" in approved.columns else None
    if by_channel_col:
        by_channel = approved.groupby(by_channel_col).apply(kpi_group).apply(
            pd.Series
        ).reset_index()
        by_channel.rename(columns={by_channel_col: "segment"}, inplace=True)
    else:
        by_channel = pd.DataFrame()

    by_cohort = approved.groupby("origination_quarter").apply(kpi_group).apply(
        pd.Series
    ).reset_index()
    by_cohort.rename(columns={"origination_quarter": "segment"}, inplace=True)

    log.info("[KPIs] Portfolio KPIs computed.")
    log.info("[KPIs] Overall — Default rate: %.2f%% | NPA: %.2f%% | Collection eff: %.2f%%",
             overall["default_rate_pct"].iloc[0],
             overall["npa_ratio_pct"].iloc[0],
             overall["collection_efficiency"].iloc[0])

    return {
        "overall_kpis":     overall,
        "by_loan_type":     by_loan_type,
        "by_risk_grade":    by_risk,
        "by_channel":       by_channel,
        "by_cohort_quarter":by_cohort,
    }


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — ML DATASET PREPARATION
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

# Feature sets for each ML task
ML_FEATURE_SETS = {
    "default_prediction": [
        # Customer risk
        "age", "monthly_income", "credit_score", "credit_history_length",
        "income_stability_score", "bank_balance_avg", "existing_loans",
        "digital_engagement_score", "debt_to_income_ratio",
        "emi_to_income_ratio", "leverage_score", "financial_stress_index",
        "credit_stability_index", "income_volatility_proxy",
        "digital_trust_score",
        # Loan attributes
        "loan_amount", "interest_rate", "loan_tenure_months",
        "loan_age_days", "is_stress_period",
        # Repayment history (if available)
        "avg_delay_days", "max_delay_days", "missed_payment_ratio",
        "delinquency_frequency", "bounce_frequency",
        "payment_regularization_score", "rolling_dpd_trend",
        # Behavioral
        "spending_volatility_index", "balance_instability_score",
        "spending_shock_frequency", "nighttime_risk_score",
        "behavioral_deterioration_score",
        "cashflow_consistency_score_mean",
        # Early warning
        "early_warning_risk_score",
    ],
    "delinquency_prediction": [
        "credit_score", "income_stability_score", "monthly_income",
        "avg_delay_days", "missed_payment_ratio", "rolling_dpd_trend",
        "worst_delinquency_stage", "bounce_frequency",
        "spending_volatility_index", "behavioral_deterioration_score",
        "spending_shock_frequency", "cashflow_consistency_score_mean",
        "app_usage_mean", "financial_stress_index",
        "early_warning_risk_score", "is_stress_period",
    ],
    "approval_decision": [
        "age", "monthly_income", "credit_score", "credit_history_length",
        "existing_loans", "debt_to_income_ratio", "leverage_score",
        "loan_amount", "interest_rate", "loan_tenure_months",
        "income_volatility_proxy", "digital_trust_score",
        "financial_stress_index",
    ],
    "segmentation": [
        "age", "monthly_income", "credit_score", "income_stability_score",
        "digital_engagement_score", "debt_to_income_ratio",
        "financial_stress_index", "credit_stability_index",
        "spending_volatility_index", "balance_instability_score",
        "early_warning_risk_score",
    ],
}

CATEGORICAL_COLS = [
    "loan_type", "employment_type", "urban_semiurban_flag",
    "gender", "origination_risk_grade", "acquisition_channel",
    "device_type", "repayment_frequency",
]

TARGET_COLS = {
    "default_prediction":    "default_flag",
    "delinquency_prediction":"worst_delinquency_stage",
    "approval_decision":     "approval_status",
}


def encode_categoricals(df: pd.DataFrame,
                          cat_cols: list = None) -> pd.DataFrame:
    """
    One-hot encode all low-cardinality categorical columns.
    Drop-first to avoid multicollinearity.
    Returns df with encoded columns and drops originals.
    """
    cat_cols = cat_cols or [c for c in CATEGORICAL_COLS if c in df.columns]
    # Only encode if cardinality <= 20 (avoid explosion for high-cardinality)
    safe_cols = [c for c in cat_cols if df[c].nunique() <= 20]
    df = pd.get_dummies(df, columns=safe_cols, drop_first=True, dtype=int)
    log.info("[ML Prep] Encoded %d categorical columns.", len(safe_cols))
    return df


def winsorise_features(df: pd.DataFrame, numeric_cols: list) -> pd.DataFrame:
    """Winsorise numeric features at 1st/99th percentile before scaling."""
    for col in numeric_cols:
        if col in df.columns:
            df[col] = winsorise(df[col])
    return df


def prepare_ml_dataset(master: pd.DataFrame, task: str = "default_prediction") -> dict:
    log.info("[ML Prep] Preparing dataset for task: %s", task)

    target_col = TARGET_COLS.get(task)
    if not target_col or target_col not in master.columns:
        log.warning("[ML Prep] Target column '%s' not found.", target_col)
        return {}

    feat_cols = [f for f in ML_FEATURE_SETS.get(task, []) if f in master.columns]

    # Handle duplicate target columns gracefully by picking the first instance
    if isinstance(master[target_col], pd.DataFrame):
        log.warning("[ML Prep] Duplicate target column '%s' found. Slicing first instance.", target_col)
        target_series = master[target_col].iloc[:, 0].copy()
    else:
        target_series = master[target_col].copy()

    # Isolate unique target name to avoid layout fragmentation
    target_name = "target_label"
    
    work = master[feat_cols + ["origination_date"]].copy()
    work[target_name] = target_series

    # Handle categorical encoding on feature set safely
    work = encode_categoricals(work, [c for c in CATEGORICAL_COLS if c in feat_cols])

    # Refresh feature columns list post-encoding
    feat_cols = [c for c in work.columns if c not in [target_name, "origination_date"]]

    # ── Safe Target Encoding ─────────────────────────────────────────────────
    if target_col == "approval_status":
        work[target_name] = (work[target_name] == "Approved").astype(int)
    elif target_col == "worst_delinquency_stage":
        work[target_name] = (work[target_name].astype(float) >= 2.0).astype(int)

    # Drop rows with missing target or features
    work = work.dropna(subset=[target_name])

    # Time-aware split
    work = work.sort_values("origination_date")
    split_idx = int(len(work) * (1 - CFG["test_size"]))
    train_df  = work.iloc[:split_idx]
    test_df   = work.iloc[split_idx:]

    X_train = train_df[feat_cols].fillna(0)
    X_test  = test_df[feat_cols].fillna(0)
    y_train = train_df[target_name].astype(int)
    y_test  = test_df[target_name].astype(int)

    # Winsorization 
    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    for col in num_cols:
        lo, hi = X_train[col].quantile(0.01), X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lo, hi)
        X_test[col]  = X_test[col].clip(lo, hi)

    # Robust Scaling
    scaler = RobustScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feat_cols, index=X_train.index)
    X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=feat_cols, index=X_test.index)

    # Calculate scalar baseline positive rate safely
    pos_rate = float(y_train.sum() / len(y_train)) * 100

    log.info("[ML Prep] Task=%s | Train=%d | Test=%d | Features=%d | Target positive rate=%.2f%%",
             task, len(X_train), len(X_test), len(feat_cols), pos_rate)

    return {
        "X_train":       X_train_scaled,
        "X_test":        X_test_scaled,
        "y_train":       y_train,
        "y_test":        y_test,
        "X_train_raw":   X_train,
        "X_test_raw":    X_test,
        "feature_names": feat_cols,
        "scaler":        scaler,
        "task":          task,
    }


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — VISUAL ANALYTICS (matplotlib — works headless)
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def plot_missing_values(df: pd.DataFrame, title: str,
                         save_as: str = None) -> None:
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if miss.empty:
        log.info("[Viz] No missing values in %s.", title)
        return
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss) * 0.4)))
    miss.plot(kind="barh", ax=ax, color="steelblue")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_title(f"Missing Values — {title}")
    ax.set_xlabel("Missing Rate")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=120)
    plt.close()


def plot_risk_distribution(master: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Risk Distribution Dashboard", fontsize=14, fontweight="bold")

    # 1. Credit score histogram
    ax = axes[0, 0]
    master["credit_score"].dropna().hist(bins=40, ax=ax, color="steelblue",
                                          edgecolor="white")
    ax.set_title("Credit Score Distribution")
    ax.set_xlabel("Credit Score"); ax.set_ylabel("Count")

    # 2. Default rate by risk grade
    ax = axes[0, 1]
    if "default_flag" in master.columns and "origination_risk_grade" in master.columns:
        dr = master.groupby("origination_risk_grade")["default_flag"].mean() * 100
        dr.sort_index().plot(kind="bar", ax=ax, color="tomato", rot=0)
        ax.set_title("Default Rate by Risk Grade")
        ax.set_ylabel("Default Rate (%)")

    # 3. Loan amount by loan type
    ax = axes[1, 0]
    if "loan_type" in master.columns:
        for lt, grp in master.groupby("loan_type"):
            grp["loan_amount"].dropna().apply(np.log1p).hist(
                bins=30, ax=ax, alpha=0.6, label=lt
            )
        ax.set_title("Log(Loan Amount) by Loan Type")
        ax.legend(fontsize=8)

    # 4. Financial stress index distribution
    ax = axes[1, 1]
    if "financial_stress_index" in master.columns:
        master["financial_stress_index"].dropna().hist(
            bins=40, ax=ax, color="mediumseagreen", edgecolor="white"
        )
        ax.set_title("Financial Stress Index Distribution")

    plt.tight_layout()
    path = os.path.join(CFG["figures_dir"], "risk_distribution.png")
    plt.savefig(path, dpi=120)
    plt.close()
    log.info("[Viz] Saved risk_distribution.png")


def plot_delinquency_trend(repayments: pd.DataFrame) -> None:
    r = repayments.copy()
    r["due_date"] = pd.to_datetime(r["due_date"])
    r["quarter"]  = r["due_date"].dt.to_period("Q").astype(str)
    stage_ord = {"Current": 0, "DPD_30": 1, "DPD_60": 2, "DPD_90": 3, "Write-Off": 4}
    r["stage_rank"] = r["delinquency_stage"].map(stage_ord)

    trend = r.groupby("quarter")["stage_rank"].mean().reset_index()
    trend.columns = ["quarter", "avg_stage_rank"]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(trend["quarter"], trend["avg_stage_rank"],
            marker="o", color="tomato", linewidth=2)
    ax.set_title("Average Delinquency Stage Trend by Quarter")
    ax.set_xlabel("Quarter"); ax.set_ylabel("Avg Stage Rank (0=Current, 4=WriteOff)")
    ax.tick_params(axis="x", rotation=45)
    # Annotate stress periods
    stress_quarters = ["2022Q2", "2022Q3", "2023Q4"]
    for q in stress_quarters:
        if q in trend["quarter"].values:
            idx = trend[trend["quarter"] == q].index[0]
            ax.axvline(x=idx, color="orange", linestyle="--", alpha=0.5)
    plt.tight_layout()
    path = os.path.join(CFG["figures_dir"], "delinquency_trend.png")
    plt.savefig(path, dpi=120)
    plt.close()
    log.info("[Viz] Saved delinquency_trend.png")


def plot_correlation_heatmap(master: pd.DataFrame) -> None:
    num_feats = [
        "credit_score", "monthly_income", "debt_to_income_ratio",
        "financial_stress_index", "emi_to_income_ratio",
        "spending_volatility_index", "behavioral_deterioration_score",
        "early_warning_risk_score", "default_flag",
    ]
    cols = [c for c in num_feats if c in master.columns]
    if len(cols) < 4:
        return
    corr = master[cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr, cmap="RdYlGn_r", vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(cols, fontsize=8)
    ax.set_title("Feature Correlation Heatmap")
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center",
                    va="center", fontsize=6, color="black")
    plt.tight_layout()
    path = os.path.join(CFG["figures_dir"], "correlation_heatmap.png")
    plt.savefig(path, dpi=120)
    plt.close()
    log.info("[Viz] Saved correlation_heatmap.png")


def plot_channel_performance(kpis: dict) -> None:
    ch = kpis.get("by_channel")
    if ch is None or ch.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ch_s = ch.sort_values("default_rate_pct")
    axes[0].barh(ch_s["segment"], ch_s["default_rate_pct"], color="tomato")
    axes[0].set_title("Default Rate by Acquisition Channel")
    axes[0].set_xlabel("Default Rate (%)")
    axes[1].barh(ch_s["segment"], ch_s["avg_clv"], color="steelblue")
    axes[1].set_title("Avg CLV by Acquisition Channel")
    axes[1].set_xlabel("Customer Lifetime Value (₹)")
    plt.tight_layout()
    path = os.path.join(CFG["figures_dir"], "channel_performance.png")
    plt.savefig(path, dpi=120)
    plt.close()
    log.info("[Viz] Saved channel_performance.png")


def run_visual_analytics(data: dict, master: pd.DataFrame, kpis: dict) -> None:
    log.info("=" * 60)
    log.info("STEP 10 — VISUAL ANALYTICS")
    log.info("=" * 60)

    plot_missing_values(data["customers"], "Customer Profile",
                        os.path.join(CFG["figures_dir"], "missing_customers.png"))
    plot_missing_values(data["loans"], "Loan Application",
                        os.path.join(CFG["figures_dir"], "missing_loans.png"))
    plot_risk_distribution(master)
    plot_delinquency_trend(data["repayments"])
    plot_correlation_heatmap(master)
    plot_channel_performance(kpis)
    log.info("Visual analytics complete.  Figures saved to %s", CFG["figures_dir"])


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — EXPORT & PERSISTENCE
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def export_all(data: dict, master: pd.DataFrame, kpis: dict,
               ml_datasets: dict) -> None:
    log.info("=" * 60)
    log.info("STEP 11 — EXPORT")
    log.info("=" * 60)

    out = CFG["output_dir"]

    # ── Cleaned tables ────────────────────────────────────────────────────────
    for name, df in [
        ("cleaned_customers", data["customers"]),
        ("cleaned_loans",     data["loans"]),
        ("cleaned_repayments",data["repayments"]),
        ("cleaned_behavioral",data["behavioral"]),
        ("cleaned_outcomes",  data["outcomes"]),
    ]:
        path = os.path.join(out, f"{name}.csv")
        df.to_csv(path, index=False)
        log.info("  Exported %-35s  %s rows  %.1f MB",
                 name + ".csv", f"{len(df):,}",
                 os.path.getsize(path) / 1e6)

    # ── Master feature table ──────────────────────────────────────────────────
    master_path = os.path.join(out, "master_feature_table.csv")
    master.to_csv(master_path, index=False)
    log.info("  Exported master_feature_table.csv  %s rows  %.1f MB",
             f"{len(master):,}", os.path.getsize(master_path) / 1e6)

    # ── KPI tables ────────────────────────────────────────────────────────────
    for kpi_name, kpi_df in kpis.items():
        if kpi_df is not None and not kpi_df.empty:
            p = os.path.join(out, f"kpi_{kpi_name}.csv")
            kpi_df.to_csv(p, index=False)
            log.info("  Exported kpi_%-25s  %s rows", kpi_name + ".csv",
                     f"{len(kpi_df):,}")

    # ── ML datasets ───────────────────────────────────────────────────────────
    for task, ml in ml_datasets.items():
        if not ml:
            continue
        task_dir = os.path.join(out, "ml", task)
        Path(task_dir).mkdir(parents=True, exist_ok=True)
        ml["X_train"].to_csv(os.path.join(task_dir, "X_train.csv"), index=False)
        ml["X_test"].to_csv( os.path.join(task_dir, "X_test.csv"),  index=False)
        ml["y_train"].to_csv(os.path.join(task_dir, "y_train.csv"), index=False)
        ml["y_test"].to_csv( os.path.join(task_dir, "y_test.csv"),  index=False)
        # Save feature list
        with open(os.path.join(task_dir, "feature_names.json"), "w") as f:
            json.dump(ml["feature_names"], f, indent=2)
        log.info("  Exported ML dataset: %-30s  train=%d  test=%d  features=%d",
                 task, len(ml["X_train"]), len(ml["X_test"]),
                 len(ml["feature_names"]))

    # ── Parquet optimised exports (requires pyarrow) ──────────────────────────
    try:
        master.to_parquet(
            os.path.join(out, "master_feature_table.parquet"),
            index=False, compression="snappy"
        )
        log.info("  Exported Parquet: master_feature_table.parquet")
    except Exception as e:
        log.info("  Parquet export skipped (%s). Install pyarrow.", str(e)[:60])

    log.info("Export complete. All files in '%s/'", out)


# ─────────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — PIPELINE ORCHESTRATOR (main entry point)
# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def print_pipeline_summary(master: pd.DataFrame, kpis: dict,
                             val_report: pd.DataFrame) -> None:
    print("\n" + "═" * 70)
    print("  MODULE 2 — PIPELINE SUMMARY")
    print("═" * 70)

    overall = kpis["overall_kpis"].iloc[0]
    print(f"\n  📊 Portfolio Overview")
    print(f"     Total Loans        : {overall['total_loans']:,}")
    print(f"     Total Disbursed    : ₹{overall['total_disbursed_INR']:,.0f}")
    print(f"     Default Rate       : {overall['default_rate_pct']}%")
    print(f"     NPA Ratio          : {overall['npa_ratio_pct']}%")
    print(f"     Collection Eff.    : {overall['collection_efficiency']}%")
    print(f"     Avg Interest Rate  : {overall['avg_interest_rate']}%")

    print(f"\n  🛠  Feature Engineering")
    print(f"     Master Table Shape : {master.shape}")
    print(f"     Total Features     : {master.shape[1]}")

    fails = val_report[val_report["status"] == "FAIL"]
    print(f"\n  ✅ Validation")
    print(f"     Total Checks       : {len(val_report)}")
    print(f"     Failed Checks      : {len(fails)}")

    print("\n  📂 Outputs saved to:", CFG["output_dir"])
    print("═" * 70 + "\n")


def run_pipeline() -> dict:
    """
    Execute the complete Module 2 pipeline end-to-end.
    Returns all artefacts in a dict for notebook exploration.
    """

    # ── Step 1: Load ──────────────────────────────────────────────────────────
    data = load_all_data()

    # ── Step 2: Clean ─────────────────────────────────────────────────────────
    data = clean_all(data)

    # ── Step 3: Validate ──────────────────────────────────────────────────────
    val_report = run_validation(data)

    # ── Step 4: Impute ────────────────────────────────────────────────────────
    data = run_imputation(data)

    # ── Step 5: Outlier detection ─────────────────────────────────────────────
    data = run_outlier_detection(data)

    # ── Step 6a: Customer risk features ──────────────────────────────────────
    log.info("=" * 60)
    log.info("STEP 6 — FEATURE ENGINEERING")
    log.info("=" * 60)
    data["customers"] = engineer_customer_risk_features(
        data["customers"], data["loans"]
    )

    # ── Step 6b: Repayment features ───────────────────────────────────────────
    rep_feats = engineer_repayment_features(data["repayments"])

    # ── Step 6c: Behavioral features ─────────────────────────────────────────
    beh_feats = engineer_behavioral_features(data["behavioral"])

    # ── Step 6d: Early warning features (adds to customers) ──────────────────
    data["customers"] = engineer_early_warning_features(
        data["customers"], rep_feats, beh_feats
    )

    # ── Step 6e: Temporal + portfolio features on loans ───────────────────────
    loan_feats = engineer_temporal_features(data["loans"], data["repayments"])
    loan_feats = engineer_portfolio_features(loan_feats, data["outcomes"])

    # ── Step 7: Assemble master ───────────────────────────────────────────────
    log.info("=" * 60)
    log.info("STEP 7 — MASTER TABLE ASSEMBLY")
    log.info("=" * 60)
    master = assemble_master_table(data, rep_feats, beh_feats, loan_feats)

    # ── Step 8: Portfolio KPIs ────────────────────────────────────────────────
    kpis = compute_portfolio_kpis(master)

    # ── Step 9: ML datasets ───────────────────────────────────────────────────
    log.info("=" * 60)
    log.info("STEP 9 — ML DATASET PREPARATION")
    log.info("=" * 60)
    ml_datasets = {}
    for task in ["default_prediction", "delinquency_prediction",
                 "approval_decision"]:
        ml_datasets[task] = prepare_ml_dataset(master, task)

    # ── Step 10: Visualisations ───────────────────────────────────────────────
    run_visual_analytics(data, master, kpis)

    # ── Step 11: Export ───────────────────────────────────────────────────────
    export_all(data, master, kpis, ml_datasets)

    # ── Summary ───────────────────────────────────────────────────────────────
    print_pipeline_summary(master, kpis, val_report)

    log.info("Module 2 pipeline finished successfully.")

    return {
        "data":        data,
        "master":      master,
        "rep_feats":   rep_feats,
        "beh_feats":   beh_feats,
        "loan_feats":  loan_feats,
        "kpis":        kpis,
        "ml_datasets": ml_datasets,
        "val_report":  val_report,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    artefacts = run_pipeline()

# ─────────────────────────────────────────────────────────────────────────────
# CELL — NOTEBOOK EXPLORATION SNIPPETS (uncomment in notebook cells)
# ─────────────────────────────────────────────────────────────────────────────
#
# Run pipeline:
# artefacts = run_pipeline()
#
# Access master table:
# master = artefacts["master"]
# master.head()
#
# Portfolio KPIs:
# artefacts["kpis"]["overall_kpis"]
# artefacts["kpis"]["by_risk_grade"]
# artefacts["kpis"]["by_loan_type"]
#
# ML dataset for default prediction:
# ml = artefacts["ml_datasets"]["default_prediction"]
# ml["X_train"].shape, ml["y_train"].value_counts()
#
# Validation report:
# artefacts["val_report"]
#
# Correlation of engineered features with default:
# master[["financial_stress_index","credit_stability_index",
#         "early_warning_risk_score","default_flag"]].corr()
#
# Feature importance (quick check with RandomForest):
# from sklearn.ensemble import RandomForestClassifier
# ml = artefacts["ml_datasets"]["default_prediction"]
# rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# rf.fit(ml["X_train"], ml["y_train"])
# imp = pd.Series(rf.feature_importances_, index=ml["feature_names"])
# imp.sort_values(ascending=False).head(20).plot(kind="barh")


12:28:32 | INFO     | Module 2 pipeline started cleanly.
12:28:32 | INFO     | ============================================================
12:28:32 | INFO     | STEP 1 — DATA LOADING
12:28:32 | INFO     | ============================================================
12:28:32 | INFO     | Loaded Customer Profile                     rows=25000     cols=19
12:28:32 | INFO     | Loaded Loan Application                     rows=65000     cols=14
12:28:34 | INFO     | Loaded Repayment Behavior                   rows=728384    cols=11
12:28:36 | INFO     | Loaded Behavioral Signals                   rows=1409893   cols=11
12:28:36 | INFO     | Loaded Outcomes                             rows=24599     cols=9
12:28:37 | INFO     | Data loading complete.
12:28:37 | INFO     | ============================================================
12:28:37 | INFO     | STEP 2 — DATA CLEANING
12:28:37 | INFO     | ============================================================
12:28:38 | INFO     | [Customers]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\abhir\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\abhir\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 63: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\abhir\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\abhir\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  

12:28:39 | INFO     |   [OK]   No orphan repayment records                             count=0
12:28:39 | INFO     |   [OK]   Valid delinquency_stage values                          count=0
12:28:39 | INFO     |   [OK]   default_flag is binary (0/1)                            count=0
12:28:39 | INFO     |   [WARN] Default rate in [5%, 25%] (actual=1.9%)                 count=1
12:28:39 | INFO     |   [OK]   No duplicate loan_id                                    count=0
12:28:39 | INFO     |   [OK]   No missing values in customers.customer_id              count=0
12:28:39 | INFO     |   [OK]   No missing values in customers.employment_type          count=0
12:28:39 | INFO     |   [OK]   No missing values in loans.loan_type                    count=0
12:28:39 | INFO     | Validation complete: 14 checks | 3 FAILED
12:28:39 | INFO     | ============================================================
12:28:39 | INFO     | STEP 4 — MISSING VALUE IMPUTATION
12:28:39 | INFO     | ===============